# Inference in Multiple Linear Regression (application based, not theoretical)
***

We'll need Numpy, Matplotlib, Pandas, and scipy.stats for this notebook, so let's load them. 

In [5]:
import numpy as np 
from scipy import stats
import statsmodels.api as sm 
import pandas as pd
import matplotlib.pylab as plt 
%matplotlib inline

### Exercise 1 - The Problem of Multiple Comparisons 
*** 

In this exercise we'll explore the **Problem of Multiple Comparisons**.  In multiple linear regression, the PMC rears its ugly head when you try to perform inference on a large number of parameters based on a single data set.  In particular, it tells us that if we look for a statistically significant relationship between a large number of features of a dataset and the response, we're very likely to find evidence that at least one of the features is important just by random chance, even in the case that none of the features are important. This is what motivates us to use things like the full and partial $F$-tests instead. 

We'll explore this phenomenon using the data in pmc_data.csv. The data contains $n=200$ observations for a response $y$ and $p=20$ features $X_1, X_2, \ldots, X_{20}$.  Load this data into a Pandas DataFrame. 

In [6]:
dfPMC = pd.read_csv("pmc_data.csv")
dfPMC.head()

,y,X1,X2,X3,X4,X5,X6,X7,X8,X9,...,X11,X12,X13,X14,X15,X16,X17,X18,X19,X20
0,2.737697,2.488435,1.887894,0.037353,1.639701,1.419009,0.101901,1.483266,0.804555,0.930170,...,2.482633,2.355615,0.642479,3.851602,0.573005,2.260237,0.655312,2.323467,0.740312,1.188648
1,-2.060299,1.750911,0.811138,0.571626,2.343696,0.702571,3.713269,0.366964,0.234172,1.379765,...,1.193246,1.165506,0.766173,0.234528,0.976693,1.650476,0.852885,2.178888,0.438472,0.865781
2,-1.001194,3.141434,0.895804,1.160973,2.047303,2.501465,1.932058,0.697282,0.400128,2.915288,...,0.210874,1.070302,0.751022,2.676697,0.167176,1.456459,0.672682,0.972043,2.292891,0.211013
3,-1.347879,3.119903,1.565272,1.861573,1.806319,0.295549,1.568728,0.960963,0.493314,4.649757,...,0.442146,1.548280,0.055933,0.503651,0.080741,0.257804,0.305269,0.430674,1.759105,0.611828
4,6.085894,1.090370,1.148386,0.677994,2.974993,0.615081,3.357223,1.287853,0.501881,0.286477,...,1.477053,2.497704,0.618649,2.430604,0.226835,2.739550,0.497371,0.857221,3.967444,1.532375


**Part A**: Let's start by doing some graphical exploration.  To look for relationships between an individual feature and the response we can make scatter plots.  Try several different features and see if any of them looks like they have a real relationship with the response. 

In [ ]:
# Extract the response and a single feature 
y, xk = dfPMC["y"], dfPMC["X10"]

# Make Scatterplot 
fig, ax = plt.subplots(nrows=1,ncols=1,figsize=(10,8))
ax.scatter(xk, y, color="steelblue", s=100, alpha=0.75)
ax.set_xlabel("x",fontsize=16)
ax.set_xlabel("y",fontsize=16)
ax.grid(alpha=0.25)

It does not appear that any of the features are related to the response. 

**Part B**: Did you find any relevant relationships?  Probably not; because there aren't any.  In fact, the response $y$ was generated using the model 

$$
Y = \epsilon \quad \textrm{where} \quad \epsilon \sim N(0,\sigma^2)
$$

OK, but pretend we don't know this.  Let's construct a multiple linear regression on the data and examine the hypothesis tests for the individual parameters provided by statsmodel's summary feature. 

In [ ]:
# Extract response vector y and feature matrix X from DataFrame
y, X = dfPMC["y"], dfPMC.iloc[:,1:]

# Add constant to X 
X = sm.add_constant(X)

# Fit OLS model 
model = sm.OLS(y, X).fit() 

# Print model summary 
model.summary()

**Part C**: Examine the p-values and 95% confidence intervals associated with each model parameter.  Do the tests indicate that any of the features have a statistically significant relationship with the response? 

The computed p-values associated with $X_2$ and $X_{19}$ are below the $\alpha = 0.05$ significance level, despite the fact that there is no such relationship in the model.  This is an illustration of the **Problem of Multiple Comparisons**.  Since there is a Type 1 error rate of $\alpha = 0.05$ associated with each test, the fact that we perform many such tests makes it extremely likely that at least one feature will incorrectly appear as important just due to random chance. 

**Part D**: Now, look at the $F$-statistic and associated p-value for the MLR model.  Do these values indicate that at least one of the features has a statistically significant relationship with the response? How can you reconcile the result of **Part C** with this result? 

The $F$-statistic is $F = 1.091$ and has an associated p-value of $0.363$.  Since this p-value is above $\alpha = 0.05$ we fail to reject the null hypothesis and conclude that none of the features have an effect on the response. 

Special thanks to Chris Ketelsen